# B2 - GAN on MNIST (generator with BatchNorm)
Generator: z(100) -> Dense256 ReLU -> BN -> Dense512 ReLU -> BN -> Dense784 Tanh -> Reshape(28,28,1)

Discriminator: Flatten -> Dense512 -> LeakyReLU(0.2) -> Dropout(0.3) -> Dense256 -> LeakyReLU(0.2) -> Dropout(0.3) -> Dense1 sigmoid

Loss: BCE, optimizer: Adam, about 10 epochs

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

In [ ]:
# 1. Load MNIST and normalize to [-1, 1]
(x_train, _), _ = keras.datasets.mnist.load_data()
x_train = (x_train.astype("float32") - 127.5) / 127.5
x_train = x_train.reshape(-1, 28, 28, 1)
BATCH, LATENT, EPOCHS = 128, 100, 10
dataset = tf.data.Dataset.from_tensor_slices(x_train).shuffle(60000).batch(BATCH, drop_remainder=True)
print(x_train.shape, x_train.min(), x_train.max())

In [ ]:
# 2. Generator and discriminator
class GAN:
    def __init__(self):
        self.generator = keras.Sequential([
            layers.Input((LATENT,)),
            layers.Dense(256, activation='relu'),
            layers.BatchNormalization(),
            layers.Dense(512, activation='relu'),
            layers.BatchNormalization(),
            layers.Dense(784, activation='tanh'),
            layers.Reshape((28, 28, 1))
        ], name='generator')

        self.discriminator = keras.Sequential([
            layers.Input((28, 28, 1)),
            layers.Flatten(),
            layers.Dense(512), layers.LeakyReLU(0.2), layers.Dropout(0.3),
            layers.Dense(256), layers.LeakyReLU(0.2), layers.Dropout(0.3),
            layers.Dense(1, activation='sigmoid')
        ], name='discriminator')

        self.bce = keras.losses.BinaryCrossentropy()
        self.g_opt = keras.optimizers.Adam(2e-4, beta_1=0.5)
        self.d_opt = keras.optimizers.Adam(2e-4, beta_1=0.5)

    @tf.function
    def train_step(self, real):
        z = tf.random.normal((BATCH, LATENT))
        with tf.GradientTape() as d_tape, tf.GradientTape() as g_tape:
            fake = self.generator(z, training=True)
            d_real = self.discriminator(real, training=True)
            d_fake = self.discriminator(fake, training=True)
            d_loss = self.bce(tf.ones_like(d_real), d_real) + self.bce(tf.zeros_like(d_fake), d_fake)
            g_loss = self.bce(tf.ones_like(d_fake), d_fake)
        d_grads = d_tape.gradient(d_loss, self.discriminator.trainable_variables)
        g_grads = g_tape.gradient(g_loss, self.generator.trainable_variables)
        self.d_opt.apply_gradients(zip(d_grads, self.discriminator.trainable_variables))
        self.g_opt.apply_gradients(zip(g_grads, self.generator.trainable_variables))
        return d_loss, g_loss

gan = GAN()
gan.generator.summary()
gan.discriminator.summary()

In [ ]:
# 3. Train
g_hist, d_hist = [], []
for epoch in range(EPOCHS):
    d_l, g_l = [], []
    for real in dataset:
        d_loss, g_loss = gan.train_step(real)
        d_l.append(d_loss); g_l.append(g_loss)
    d_hist.append(np.mean(d_l)); g_hist.append(np.mean(g_l))
    print(f"Epoch {epoch+1}/{EPOCHS}: D loss={d_hist[-1]:.4f}  G loss={g_hist[-1]:.4f}")

In [ ]:
# 4. Plot losses and show a 5x5 grid of generated digits
plt.plot(d_hist, label='Discriminator'); plt.plot(g_hist, label='Generator')
plt.xlabel('epoch'); plt.ylabel('loss'); plt.legend(); plt.title('GAN losses'); plt.show()

imgs = gan.generator(tf.random.normal((25, LATENT)), training=False).numpy()
imgs = (imgs + 1) / 2.0
plt.figure(figsize=(5, 5))
for i in range(25):
    plt.subplot(5, 5, i + 1); plt.imshow(imgs[i, :, :, 0], cmap='gray'); plt.axis('off')
plt.suptitle('Generated digits'); plt.show()